# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [9]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_are

# Read the supplied tab-separated files.
# The first column contains row labels, so exclude it from the numeric data.
state_columns = ["x", "v", "a"]

A = pd.read_csv("model-A.tsv", sep="\t", index_col=0).loc[
    state_columns, state_columns
].to_numpy(dtype=float)

B = pd.read_csv("model-B.tsv", sep="\t", index_col=0).loc[
    state_columns, ["u"]
].to_numpy(dtype=float)

Q = pd.read_csv("cost-Q.tsv", sep="\t", index_col=0).loc[
    state_columns, state_columns
].to_numpy(dtype=float)

R = pd.read_csv("cost-R.tsv", sep="\t", index_col=0).loc[
    ["u"], ["u"]
].to_numpy(dtype=float)

# The system follows: next_state = A @ state + B @ action.
# Q penalizes deviations in the state; R penalizes control effort.
# Solve the discrete algebraic Riccati equation for the
# infinite-horizon optimal cost matrix P.
P = solve_discrete_are(A, B, Q, R)

# Compute the optimal feedback gain:
# K = inverse(R + B.T @ P @ B) @ B.T @ P @ A
# np.linalg.solve avoids explicitly calculating a matrix inverse.
K = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)

# The controller uses action = -K @ state.
# Save K itself; the negative sign is applied when calculating actions.
print("K (columns: x, v, a):")
print(K)

# A stable discrete-time controller has closed-loop eigenvalues
# whose magnitudes are all less than 1.
eigenvalues = np.linalg.eigvals(A - B @ K)
print("Closed-loop eigenvalue magnitudes:", np.abs(eigenvalues))
assert np.all(np.abs(eigenvalues) < 1)

K (columns: x, v, a):
[[0.33445985 1.30445413 1.85813088]]
Closed-loop eigenvalue magnitudes: [0.36935751 0.55032663 0.55032663]


Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [10]:
# YOUR CHANGES HERE


control_K_intended = pd.DataFrame(K, columns=["x", "v", "a"])
control_K_intended.to_csv(
    "control-K-intended.tsv",
    sep="\t",
    index=False
)

print(control_K_intended)

         x         v         a
0  0.33446  1.304454  1.858131


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [12]:
# YOUR CHANGES HERE

qc_log = pd.read_csv("qc-log.tsv", sep="\t")

# Select the state columns in the same order used to calculate K.
# Each row represents the drone's state at one recorded time.
states = qc_log[["x", "v", "a"]].to_numpy(dtype=float)

# Recalculate the action for every logged state using u = -K @ state.
# Since our states are stored as rows, use states @ K.T.
# This gives one action per row; ravel() converts it to a 1D array.
u_check = -(states @ K.T).ravel()

# Pair each calculated action with its original log index.
qc_check = pd.DataFrame({
    "index": qc_log["index"],
    "u_check": u_check
})

print(qc_check.head())

   index   u_check
0      0  1.672299
1      1 -1.152095
2      2 -1.235183
3      3 -0.288504
4      4  0.369202


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [13]:
# YOUR CHANGES HERE

qc_check.to_csv("qc-check.tsv", sep="\t", index=False)

print(f"Saved {len(qc_check)} recalculated actions to qc-check.tsv")

Saved 100 recalculated actions to qc-check.tsv


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [ ]:
# YOUR CHANGES HERE
qc_log = pd.read_csv("qc-log.tsv", sep="\t")
states = qc_log[["x", "v", "a"]].to_numpy(dtype=float)

# These are the actions the drone actually took.
actual_actions = qc_log["u"].to_numpy(dtype=float)

# Fit the relationship: actual_actions ≈ states @ coefficients.
# Least squares finds the coefficients that minimize squared errors.
# We do not add a column of ones, so there is no intercept.
coefficients, _, _, _ = np.linalg.lstsq(
    states, actual_actions, rcond=None
)

# The controller follows u = -K_actual @ state.
# Therefore, the fitted coefficients equal -K_actual.
# Negate them and reshape into a matrix with one row.
K_actual = -coefficients.reshape(1, -1)

print("Actual control gain K (columns: x, v, a):")
print(K_actual)

# Check how closely the estimated controller reproduces logged actions.
predicted_actions = -(states @ K_actual.T).ravel()
rmse = np.sqrt(np.mean((actual_actions - predicted_actions) ** 2))
print("Root mean squared prediction error:", rmse)

Actual control gain K (columns: x, v, a):
[[0.34043755 1.30012023 1.95011696]]
Root mean squared prediction error: 1.8586852518859711e-16


Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [15]:
# YOUR CHANGES HERE


control_K_actual = pd.DataFrame(
    K_actual,
    columns=["x", "v", "a"]
)
control_K_actual.to_csv(
    "control-K-actual.tsv",
    sep="\t",
    index=False
)

print(control_K_actual)

          x        v         a
0  0.340438  1.30012  1.950117


Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [16]:
# YOUR CHANGES HERE

qc_train = pd.read_csv("qc-train.tsv", sep="\t")

state_columns = ["x", "v", "a"]

# Each input row contains the current state and current action.
# Exclude the last row because its next state is not in the file.
inputs = qc_train[["x", "v", "a", "u"]].iloc[:-1].to_numpy(dtype=float)

# Each target row is the state from the following time step.
# Exclude the first row to align each input with its next state.
next_states = qc_train[state_columns].iloc[1:].to_numpy(dtype=float)

# Fit: next_states ≈ inputs @ coefficients.
# No intercept is included because the model is:
# next_state = A @ current_state + B @ current_action.
coefficients, _, rank, _ = np.linalg.lstsq(
    inputs, next_states, rcond=None
)

# Four independent input columns are needed to estimate their effects.
assert rank == 4, "The training inputs must have full column rank."

# The fitted coefficients have input variables along the rows and
# predicted state variables along the columns.
# Transpose the appropriate blocks to match the A and B conventions.
A_new = coefficients[:3, :].T
B_new = coefficients[3:, :].T

print("Estimated A:")
print(A_new)

print("\nEstimated B:")
print(B_new)

# Check how well the estimated model predicts the next logged states.
predicted_next_states = inputs @ coefficients
rmse = np.sqrt(np.mean(
    (next_states - predicted_next_states) ** 2,
    axis=0
))

print("\nPrediction RMSE for x, v, and a:")
print(rmse)

Estimated A:
[[ 1.00000000e+00  1.10000000e+00  6.72481647e-16]
 [ 1.05851957e-16  9.00000000e-01  9.50000000e-01]
 [-3.26396627e-16 -6.10622664e-16  1.10000000e+00]]

Estimated B:
[[ 4.70546685e-16]
 [-1.00000000e-02]
 [ 9.00000000e-01]]

Prediction RMSE for x, v, and a:
[4.47918225e-16 2.42113915e-16 1.70858015e-16]


Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [17]:
# YOUR CHANGES HERE

# Match the supplied model files:
# A has rows and columns labeled x, v, a.
# B has rows labeled x, v, a and one action column labeled u.
A_new_table = pd.DataFrame(
    A_new,
    index=state_columns,
    columns=state_columns
)

B_new_table = pd.DataFrame(
    B_new,
    index=state_columns,
    columns=["u"]
)

# Keep the row labels and name their column "index",
# matching the original model-A.tsv and model-B.tsv files.
A_new_table.to_csv(
    "model-A-new.tsv",
    sep="\t",
    index_label="index"
)

B_new_table.to_csv(
    "model-B-new.tsv",
    sep="\t",
    index_label="index"
)

print("Saved model-A-new.tsv and model-B-new.tsv")

Saved model-A-new.tsv and model-B-new.tsv


Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.